# Detecting trash in images

## Libraries

In [ ]:
#!pip install -q albumentations numpy opencv-python pycocotools tqdm ultralytics yolov8 matplotlib 
#!pip install torch torchvision torchaudio 

In [ ]:
import os
import yaml
from ultralytics import YOLO
import zipfile
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import random
from unidecode import unidecode

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
import glob

In [ ]:
import torch
print(torch.__file__)
#print("current GPU device: ",torch.cuda.current_device())
#print("Is available?: ",torch.cuda.is_available())
#print("CUDA version: ",torch.version.cuda)
#print("Number of GPUs: ",torch.cuda.device_count())

In [ ]:
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

## Custom functions

In [ ]:
# Function to zip and extract dataset
def create_zip(source_folder, destination_zip):
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zip_ref:
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                zip_ref.write(file_path, arcname=os.path.relpath(file_path, source_folder))

def extract_zip(zip_file, destination_folder):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(destination_folder)


# Function to show image with labels
def show_random_labeled_image(class_names):
    image_paths = glob.glob("train/images/*.jpg")  # Adjust format if needed
    label_paths = glob.glob("train/labels/*.txt")

    if not image_paths or not label_paths:
        print("No images or labels found.")
        return
    
    # Pick a random image
    random_image = random.choice(image_paths)
    label_path = random_image.replace("images", "labels").replace(".jpg", ".txt")


    #img = Image.fromarray(random_image)  # Convert OpenCV image to PIL image
    #draw = ImageDraw.Draw(img)
    #font = ImageFont.truetype("arial.ttf", 15)
    #draw.rectangle(position, text, font=font, fill="red")
    #draw.text(position, text, font=font, fill="red")
    #image_with_text = np.array(img)  # Convert PIL image back to OpenCV image

    #draw_predicted.rectangle([x1, y1, x2, y2], outline="blue", width=2)
    #draw_predicted.text((x1, y1), predicted_labels[i] if i < len(predicted_labels) else "Unknown", fill="blue", font=font)

    # Load image
    img = cv2.imread(random_image)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Draw labels
    h, w, _ = img.shape
    with open(label_path, "r") as f:
        for line in f.readlines():
            class_id, x, y, bw, bh = map(float, line.split())
            x, y, bw, bh = int(x * w), int(y * h), int(bw * w), int(bh * h)
            cv2.rectangle(img, (x - bw//2, y - bh//2), (x + bw//2, y + bh//2), (0, 255, 0), 2)
            cv2.putText(img, class_names[int(class_id)], (x, y-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Random Image with Labels: {random_image}")
    plt.show()


def plot_model_loss(log_data):
    # Convert necessary columns to numeric
    log_data['epoch'] = pd.to_numeric(log_data['epoch'], errors='coerce').astype(int)  # Convert to integer
    log_data['train/box_loss'] = pd.to_numeric(log_data['train/box_loss'], errors='coerce')
    log_data['train/cls_loss'] = pd.to_numeric(log_data['train/cls_loss'], errors='coerce')
    log_data['train/dfl_loss'] = pd.to_numeric(log_data['train/dfl_loss'], errors='coerce')
    log_data['val/box_loss'] = pd.to_numeric(log_data['val/box_loss'], errors='coerce')
    log_data['val/cls_loss'] = pd.to_numeric(log_data['val/cls_loss'], errors='coerce')
    log_data['val/dfl_loss'] = pd.to_numeric(log_data['val/dfl_loss'], errors='coerce')
    
    # Drop rows with NaN values in relevant columns
    log_data = log_data.dropna(subset=['epoch', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 
                                       'val/box_loss', 'val/cls_loss', 'val/dfl_loss'])
    
    # Plot the training and validation losses
    plt.figure(figsize=(12, 8))
    
    # Plot training losses
    plt.plot(log_data['epoch'], log_data['train/box_loss'], label='Train Box Loss', linestyle='-', marker='o')
    plt.plot(log_data['epoch'], log_data['train/cls_loss'], label='Train Class Loss', linestyle='-', marker='x')
    plt.plot(log_data['epoch'], log_data['train/dfl_loss'], label='Train DFL Loss', linestyle='-', marker='s')
    
    # Plot validation losses
    plt.plot(log_data['epoch'], log_data['val/box_loss'], label='Val Box Loss', linestyle='--', marker='o')
    plt.plot(log_data['epoch'], log_data['val/cls_loss'], label='Val Class Loss', linestyle='--', marker='x')
    plt.plot(log_data['epoch'], log_data['val/dfl_loss'], label='Val DFL Loss', linestyle='--', marker='s')
    
    # Customize the plot
    plt.title('Training and Validation Losses over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.xticks(ticks=range(0, len(log_data['epoch']), 100))  # Ensure that the epoch ticks are shown as integers
    # plt.xticks(log_data['epoch'])  # Ensure that the epoch ticks are shown as integers
    plt.legend(loc='upper right')
    
    # Show the plot
    plt.show()


# Function to parse ground truth annotations in YOLO format
def parse_annotation(annotation_path):
    """
    Parses a YOLO-style annotation file and extracts the class IDs and bounding box information.
    """
    if not os.path.exists(annotation_path):
        print(f"Annotation file {annotation_path} not found.")
        return [], []  # Return empty lists if annotation file is missing
    
    with open(annotation_path, 'r') as file:
        lines = file.readlines()
    
    labels = []
    boxes = []
    for line in lines:
        parts = line.strip().split()
        class_id = int(parts[0])  # Class ID
        # YOLO format: class_id, x_center, y_center, width, height
        box = [float(x) for x in parts[1:]]  # Bounding box: [x_center, y_center, width, height]
        labels.append(class_id)
        boxes.append(box)
    return labels, boxes



# Function for testing detection
def testing_detection(best_model,class_names,test_image_dir,test_label_dir,output_dir):
    # Get the list of test images
    test_images = glob.glob(os.path.join(test_image_dir, '*.jpg'))  # Adjust for correct extension if needed
    
    # Output directory to save inference results
    os.makedirs(output_dir, exist_ok=True)
    
    # Loop through each image and perform inference
    for img_path in test_images:
        # Get the corresponding annotation file (in YOLO format)
        annotation_path = os.path.join(test_label_dir, os.path.basename(img_path).replace(".jpg", ".txt").replace(".JPG", ".txt"))
        
        # Perform inference without verbose output
        results = best_model(img_path, verbose=False)  # Run YOLOv8 on the image
        
        # Get actual labels (ground truth) from annotation file
        actual_labels, actual_boxes = parse_annotation(annotation_path)
        actual_labels_names = [class_names[label] for label in actual_labels]
        
        # Save the result image with predictions
        img_name = os.path.basename(img_path)
        result_img_path = os.path.join(output_dir, img_name)
        results[0].save(result_img_path)
        
        # Extract predicted labels and bounding boxes
        if results[0].boxes is None or len(results[0].boxes.cls) == 0:
            predicted_labels = ["No prediction"]
            predicted_boxes = []
        else:
            predicted_labels = [results[0].names[int(cls)] for cls in results[0].boxes.cls]
            predicted_boxes = results[0].boxes.xywh.cpu().numpy()  # Ensure numpy format for further processing
        
        # Open the original image for proper ground truth visualization
        img_predicted = Image.open(result_img_path)
        img_actual = Image.open(img_path)  # Reload the original image for ground truth        
        
        # Create drawing objects
        draw_predicted = ImageDraw.Draw(img_predicted)
        draw_actual = ImageDraw.Draw(img_actual)
        font = ImageFont.truetype("arial.ttf", 15)
        
        # Draw predicted bounding boxes (blue) on the predicted image
        img_width, img_height = img_predicted.size

        #create file for annotation
        file_annotation = open( os.path.join(output_dir, img_name.replace(".jpg", ".txt").replace(".JPG", ".txt")) , 'w')
        print('IMAGE SIZE (width x height):',file=file_annotation)
        print('  {} {} {}'.format( img_width , " ", img_height),file=file_annotation)
        print('PREDICTION:',file=file_annotation)
        
        if len(predicted_boxes) == 0:  # Check if predicted_boxes is empty
            draw_predicted.text((10, 10), "No prediction", fill="red")
        else:
            for i, box in enumerate(predicted_boxes):
                x_center, y_center, width, height = box
                #x1 = int((x_center - width / 2) * 1)  /  img_width
                #y1 = int((y_center - height / 2) * 1)  / img_height
                #x2 = int((x_center + width / 2) * 1)  / img_width
                #y2 = int((y_center + height / 2) * 1)  / img_height

                x1 = int((x_center - width / 2) * 1)  / 1
                y1 = int((y_center - height / 2) * 1)  / 1
                x2 = int((x_center + width / 2) * 1)  / 1
                y2 = int((y_center + height / 2) * 1)  / 1
                draw_predicted.rectangle([x1, y1, x2, y2], outline="blue", width=2)
                draw_predicted.text((x1, y1), predicted_labels[i] if i < len(predicted_labels) else "Unknown", fill="blue", font=font)
                if i < len(predicted_labels): 
                    print('  {} {}'.format( class_names.index(predicted_labels[i]) , " ".join(str(x) for x in [x1,y1,x2,y2]) ),file=file_annotation)
        
        print('OBSERVATION:',file=file_annotation)
        # Draw ground truth bounding boxes (green) on the actual image
        for i, box in enumerate(actual_boxes):
            x_center, y_center, width, height = box
            x1 = int((x_center - width / 2) * img_width)
            y1 = int((y_center - height / 2) * img_height)
            x2 = int((x_center + width / 2) * img_width)
            y2 = int((y_center + height / 2) * img_height)
            draw_actual.rectangle([x1, y1, x2, y2], outline="green", width=2)
            draw_actual.text((x1, y1), actual_labels_names[i], fill="white",font=font)        
            print('  {} {}'.format( class_names.index(actual_labels_names[i]) , " ".join(str(x) for x in box) ),file=file_annotation)
    
        file_annotation.close()
        
        # Display images side by side
        fig, axes = plt.subplots(1, 2, figsize=(15, 7))
        
        axes[0].imshow(img_predicted)
        axes[0].set_title("\n".join(predicted_labels), fontsize=14, wrap=True)
        axes[0].axis("off")
        
        axes[1].imshow(img_actual)
        axes[1].set_title("\n".join(actual_labels_names), fontsize=14, wrap=True)
        axes[1].axis("off")
        
        plt.show()

## Dataset Extraction

In [ ]:
# Paths for dataset
#source_folder = '/'
#destination_zip = 'taco_dataset.zip'
#destination_folder = '/'

# Zip & Extract
#create_zip(source_folder, destination_zip)
#extract_zip(destination_zip, destination_folder)

## Dataset config

In [ ]:
# Define dataset configuration (data.yaml)
data_yaml = dict(
    train='train/images',
    val='valid/images',
    test='test/images',
    nc=18,
    names_english=['Aluminium foil', 'Bottle', 'Bottle cap', 'Broken glass', 'Can', 
           'Carton', 'Cigarette', 'Cup', 'Lid', 'Other litter', 'Other plastic', 
           'Paper', 'Plastic bag - wrapper', 'Plastic container', 'Pop tab', 
           'Straw', 'Styrofoam piece', 'Unlabeled litter'],
    names=['Papel de aluminio', 'Botella', 'Tapón de botella', 'Vidrio roto', 'Lata', 'Cartón',
           'Cigarrillo','Vaso', 'Tapa', 'Otros residuos', 'Otros pláticos', 'Papel', 
           'Bolsa de plástico / envoltorio', 'Contenedor de plástico', 'Anilla de lata', 'Pajita', 
           'Corcho blanco', 'Residuo sin clasificar']
)

# Save data.yaml file
with open('data.yaml', 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=False)

In [ ]:
import yaml

with open('data.yaml', 'r') as f:
    data_info = yaml.safe_load(f)

print(f"Number of Classes: {data_info['nc']}")
print("Classes:", data_info['names'])

## Distribution of classes in training set

In [ ]:
# Get all label files
label_paths = glob.glob("train/labels/*.txt")  # Adjust if needed
class_counts = Counter()

# Read each label file and count occurrences of each class
for path in label_paths:
    with open(path, "r") as f:
        for line in f.readlines():
            class_id = int(line.split()[0])
            class_counts[class_id] += 1

# Convert to DataFrame for visualization
class_names_english = ['Aluminium foil', 'Bottle', 'Bottle cap', 'Broken glass', 'Can', 
           'Carton', 'Cigarette', 'Cup', 'Lid', 'Other litter', 'Other plastic', 
           'Paper', 'Plastic bag - wrapper', 'Plastic container', 'Pop tab', 
           'Straw', 'Styrofoam piece', 'Unlabeled litter']

class_names = ['Papel de aluminio', 'Botella', 'Tapón de botella', 'Vidrio roto', 'Lata', 'Cartón',
           'Cigarrillo','Vaso', 'Tapa', 'Otros residuos', 'Otros pláticos', 'Papel', 
           'Bolsa de plástico / envoltorio', 'Contenedor de plástico', 'Anilla de lata', 'Pajita', 
           'Corcho blanco', 'Residuo sin clasificar']
#class_names =[x.encode('utf8') for x in class_names]
    
df = pd.DataFrame({'Class': class_names, 'Count': [class_counts[i] for i in range(18)]})

# Print the class counts
print("Class Counts:")
for i, row in df.iterrows():
    print(f"{row['Class']}: {row['Count']}")

# Plot the distribution
plt.figure(figsize=(12, 5))
sns.barplot(x="Class", y="Count", hue="Class", data=df, palette="viridis",legend=False)
plt.xticks(rotation=90)
plt.title("Class Distribution in Training Data")
plt.show()

## An example of class prediction and annotation

In [ ]:
# Run multiple times to see different samples
class_names_sin_acentos=[unidecode(x, "utf-8")  for x in class_names] 
show_random_labeled_image(class_names_sin_acentos)

## Training Parameters

In [ ]:
# Define training parameters
epochs = 40
batch_size = 32
imgsz = 640  # Image size
optimizer_type = 'AdamW'  # AdamW optimizer (recommended for better regularization)
lr = 1e-4
weight_decay = 1e-4

## Model training

In [ ]:
# Initialize YOLOv11 model (pre-trained weights)
model = YOLO("yolov8s.pt")
# Move model to GPU
model.to('cuda')

In [ ]:
# Train model with Cosine Annealing learning rate scheduler
model.train(
    data="data.yaml",
    epochs=epochs,
    device='cuda',
    batch=batch_size,
    imgsz=imgsz,
    optimizer=optimizer_type,
    lr0=lr,  # Initial learning rate
    weight_decay=weight_decay,
    save=True,  # Save the best model
    save_period=1,  # Save model every 10 epochs
    val=True  # Evaluate on validation set
    #save_dir='runs/train/exp'  # Save directory for model and logs
    #project=".", # Custom root directory
    #name="runs/train/exp" # Subdirectory for this run
)

### OJO, este proceso tarda bastante si tiene un epochs alto (ver 1.7. Training Parameters)

In [ ]:
import os

save_dir = 'runs/detect/train/weights'

# List files and directories in save_dir
if os.path.exists(save_dir):
    print(os.listdir(save_dir))
else:
    print(f"The directory '{save_dir}' does not exist.")

## Best model validation metrics

In [ ]:
best_model1 = YOLO("runs_ANTIGUO/detect/train/weights/best.pt")
val_results1 = best_model1.val()

# Modelo 2
best_model2 = YOLO("runs_GPU_PRIMERO_1000_FOTOS/detect/train/weights/best.pt")
val_results2 = best_model2.val()

# Modelo 3
best_model3 = YOLO("runs_GPU_SEGUNDO_4000_FOTOS/detect/train/weights/best.pt")
val_results3 = best_model3.val()

In [ ]:
print("===== MODELO 1: runs_ANTIGUO =====")
print(f"Mean Precision: {val_results1.box.mp:.4f}")
print(f"Mean Recall: {val_results1.box.mr:.4f}")
print(f"mAP@50: {val_results1.box.map50:.4f}")
print(f"mAP@50-95: {val_results1.box.map:.4f}")

print("\n===== MODELO 2: runs_GPU_PRIMERO_1000_FOTOS =====")
print(f"Mean Precision: {val_results2.box.mp:.4f}")
print(f"Mean Recall: {val_results2.box.mr:.4f}")
print(f"mAP@50: {val_results2.box.map50:.4f}")
print(f"mAP@50-95: {val_results2.box.map:.4f}")

print("\n===== MODELO 3: runs_GPU_SEGUNDO_4000_FOTOS =====")
print(f"Mean Precision: {val_results3.box.mp:.4f}")
print(f"Mean Recall: {val_results3.box.mr:.4f}")
print(f"mAP@50: {val_results3.box.map50:.4f}")
print(f"mAP@50-95: {val_results3.box.map:.4f}")

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Modelo": [
        "ANTIGUO",
        "GPU_1000_FOTOS",
        "GPU_4000_FOTOS"
    ],
    "Precision": [
        val_results1.box.mp,
        val_results2.box.mp,
        val_results3.box.mp
    ],
    "Recall": [
        val_results1.box.mr,
        val_results2.box.mr,
        val_results3.box.mr
    ],
    "mAP50": [
        val_results1.box.map50,
        val_results2.box.map50,
        val_results3.box.map50
    ],
    "mAP50-95": [
        val_results1.box.map,
        val_results2.box.map,
        val_results3.box.map
    ]
})

print(results_df)

## Training & validation loss

In [ ]:
%matplotlib inline
import pandas as pd

# =====================================================================
# EXPERIMENTO 1: 80 Epochs (ANTIGUO)
# =====================================================================
print("📊 RESULTADOS EXPERIMENTO 1 (80 epochs)")
log_file_1 = 'runs_ANTIGUO/detect/train/results.csv'
log_data_1 = pd.read_csv(log_file_1)

print(log_data_1.columns)
print(log_data_1.head())

plot_model_loss(log_data_1)


# =====================================================================
# EXPERIMENTO 2: 1000 Epochs (250 imágenes de validación)
# =====================================================================
print("\n📊 RESULTADOS EXPERIMENTO 2 (1000 epochs - 250 val)")
log_file_2 = 'runs_GPU_PRIMERO_1000_FOTOS/detect/train/results.csv'
log_data_2 = pd.read_csv(log_file_2)

print(log_data_2.columns)
print(log_data_2.head())

plot_model_loss(log_data_2)


# =====================================================================
# EXPERIMENTO 3: 1000 Epochs (1000 imágenes de validación - Optimizado)
# =====================================================================
print("\n📊 RESULTADOS EXPERIMENTO 3 (1000 epochs - 1000 val)")
log_file_3 = 'runs_GPU_SEGUNDO_4000_FOTOS/detect/train/results.csv'
log_data_3 = pd.read_csv(log_file_3)

print(log_data_3.columns)
print(log_data_3.head())

plot_model_loss(log_data_3)

In [ ]:
%matplotlib inline
import pandas as pd

# Read the log file into a DataFrame
log_file = 'runs/detect/train/results.csv'
log_data = pd.read_csv(log_file)

# Check the first few rows of the data and column names
print(log_data.columns)
print(log_data.head())


plot_model_loss(log_data)

## Validation Metrics plot

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Scalar values from results_dict
precision = val_results3.results_dict['metrics/precision(B)']
recall = val_results3.results_dict['metrics/recall(B)']
map50 = val_results3.results_dict['metrics/mAP50(B)']
map50_95 = val_results3.results_dict['metrics/mAP50-95(B)']

# Plotting single values (snapshot)
metrics = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
values = [precision, recall, map50, map50_95]

plt.figure(figsize=(8, 6))
plt.bar(metrics, values, color=['#BF7897', '#78B1BF', '#78BF8B', '#8C78BF'])
plt.title('Model Evaluation Metrics')
plt.ylabel('Values')
plt.show()

## Saving best model & testing

In [ ]:
#best_model = YOLO('runs/detect/train/weights/best.pt')
test_results = best_model.val(data='data.yaml', split='test')

In [ ]:
best_model_1 = YOLO('runs_ANTIGUO/detect/train/weights/best.pt')
test_results_1 = best_model_1.val(data='data.yaml', split='test')

In [ ]:
# Imprimimos sus 4 métricas en el test set
print(f"Test Precision 1: {test_results_1.box.mp:.4f}")
print(f"Test Recall 1:    {test_results_1.box.mr:.4f}")
print(f"Test mAP@50 1:    {test_results_1.box.map50:.4f}")
print(f"Test mAP@50-95 1: {test_results_1.box.map:.4f}")
print("-" * 50)

In [ ]:
print("Evaluando el Experimento 2...")
best_model_2 = YOLO('runs_GPU_PRIMERO_1000_FOTOS/detect/train/weights/best.pt')
test_results_2 = best_model_2.val(data='data.yaml', split='test')

In [ ]:
print(f"Test Precision 2: {test_results_2.box.mp:.4f}")
print(f"Test Recall 2:    {test_results_2.box.mr:.4f}")
print(f"Test mAP@50 2:    {test_results_2.box.map50:.4f}")
print(f"Test mAP@50-95 2: {test_results_2.box.map:.4f}")
print("-" * 50)

In [ ]:
print("Evaluando el Experimento 3")
best_model3 = YOLO('runs_GPU_SEGUNDO_4000_FOTOS/detect/train/weights/best.pt')
test_results3 = best_model3.val(data='data.yaml', split='test')

In [ ]:
print(f"Test Precision 3: {test_results3.box.mp:.4f}")
print(f"Test Recall 3:    {test_results3.box.mr:.4f}")
print(f"Test mAP@50 3:    {test_results3.box.map50:.4f}")
print(f"Test mAP@50-95 3: {test_results3.box.map:.4f}")
print("-" * 50)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Scalar values from results_dict
precision = test_results_3.results_dict['metrics/precision(B)']
recall = test_results_3.results_dict['metrics/recall(B)']
map50 = test_results_3.results_dict['metrics/mAP50(B)']
map50_95 = test_results_3.results_dict['metrics/mAP50-95(B)']

# Plotting single values (snapshot)
metrics = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
values = [precision, recall, map50, map50_95]

plt.figure(figsize=(8, 6))
plt.bar(metrics, values, color=['#BF7897', '#78B1BF', '#78BF8B', '#8C78BF'])
plt.title('Model Evaluation Metrics')
plt.ylabel('Values')
plt.show()

In [ ]:
# Print test metrics
print(f"Test Precision: {test_results.box.mp:.4f}")
print(f"Test Recall: {test_results.box.mr:.4f}")
print(f"Test mAP@50: {test_results.box.map50:.4f}")
print(f"Test mAP@50-95: {test_results.box.map:.4f}")

In [ ]:
# 1. Cargamos explícitamente los mejores pesos del Experimento 3 (el optimizado)
best_model = YOLO('runs_GPU_SEGUNDO_4000_FOTOS/detect/train/weights/best.pt')

# Class names 
class_names_english = ['Aluminium foil', 'Bottle', 'Bottle cap', 'Broken glass', 'Can', 'Carton', 
               'Cigarette', 'Cup', 'Lid', 'Other litter', 'Other plastic', 'Paper', 
               'Plastic bag - wrapper', 'Plastic container', 'Pop tab', 'Straw', 
               'Styrofoam piece', 'Unlabeled litter']

class_names = ['Papel de aluminio', 'Botella', 'Tapón de botella', 'Vidrio roto', 'Lata', 'Cartón',
           'Cigarrillo','Vaso', 'Tapa', 'Otros residuos', 'Otros pláticos', 'Papel', 
           'Bolsa de plástico / envoltorio', 'Contenedor de plástico', 'Anilla de lata', 'Pajita', 
           'Corcho blanco', 'Residuo sin clasificar']

# Path to the test images and corresponding labels (annotations)
test_image_dir = 'test/images/'
test_label_dir = 'test/labels/'

# Output directory to save inference results
output_dir = "detection_results3/"

testing_detection(best_model,class_names,test_image_dir,test_label_dir,output_dir)

In [ ]:
# Load the YOLO model
#best_model = YOLO('runs/detect/train/weights/best.pt')

# Class names 
class_names_english = ['Aluminium foil', 'Bottle cap', 'Bottle', 'Broken glass', 'Can', 'Carton', 
               'Cigarette', 'Cup', 'Lid', 'Other litter', 'Other plastic', 'Paper', 
               'Plastic bag - wrapper', 'Plastic container', 'Pop tab', 'Straw', 
               'Styrofoam piece', 'Unlabeled litter']

class_names = ['Papel de aluminio', 'Tapón de botella', 'Botella', 'Vidrio roto', 'Lata', 'Cartón',
           'Cigarrillo','Vaso', 'Tapa', 'Otros residuos', 'Otros pláticos', 'Papel', 
           'Bolsa de plástico / envoltorio', 'Contenedor de plástico', 'Anilla de lata', 'Pajita', 
           'Corcho blanco', 'Residuo sin clasificar']

# Path to the test images and corresponding labels (annotations)
test_image_dir = 'test/images/'
test_label_dir = 'test/labels/'

# Output directory to save inference results
output_dir = "detection_results/"

testing_detection(best_model,class_names,test_image_dir,test_label_dir,output_dir)

# Improve training

In [ ]:
epochs=80
# Load the YOLO model
model = YOLO('runs/detect/train/weights/last.pt')
# Train model 
model.train(
    data="data.yaml",
    epochs=epochs,
    device='cuda',
    batch=batch_size,
    imgsz=imgsz,
    optimizer=optimizer_type,
    lr0=lr,  # Initial learning rate
    weight_decay=weight_decay,
    save=True,  # Save the best model
    save_period=1,  # Save model every 10 epochs
    val=True  # Evaluate on validation set
)
### OJO, ahora se crean pesos en la carpeta runs/detect/train2, en lugar de runs/detect/train

In [ ]:
best_model = YOLO('runs/detect/train2/weights/best.pt')
val_results = best_model.val()

print(f"Best Validation Metrics from Best Model:")
print(f"Mean Precision: {val_results.box.mp:.4f}")  # Mean Precision
print(f"Mean Recall: {val_results.box.mr:.4f}")     # Mean Recall
print(f"mAP@50: {val_results.box.map50:.4f}")       # Mean Average Precision at IoU 0.5
print(f"mAP@50-95: {val_results.box.map:.4f}")      # Mean Average Precision at IoU 0.5-0.95

In [ ]:
# Read the log file into a DataFrame
log_file = 'runs/detect/train2/results.csv'
log_data = pd.read_csv(log_file)

# Check the first few rows of the data and column names
#print(log_data.columns)
#print(log_data.head())


plot_model_loss(log_data)

# Scalar values from results_dict
precision = val_results.results_dict['metrics/precision(B)']
recall = val_results.results_dict['metrics/recall(B)']
map50 = val_results.results_dict['metrics/mAP50(B)']
map50_95 = val_results.results_dict['metrics/mAP50-95(B)']

# Plotting single values (snapshot)
metrics = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
values = [precision, recall, map50, map50_95]

plt.figure(figsize=(9, 6))
plt.rcParams.update({'font.size': 10})
plt.bar(metrics, values, color=['b', 'r', 'g', 'purple'])
plt.title('Model Evaluation Metrics')
plt.ylabel('Values')
plt.show()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 3. Extraemos las métricas generadas
precision = test_results_3.results_dict['metrics/precision(B)']
recall = test_results_3.results_dict['metrics/recall(B)']
map50 = test_results_3.results_dict['metrics/mAP50(B)']
map50_95 = test_results_3.results_dict['metrics/mAP50-95(B)']

# 4. Creamos el gráfico con el diseño adaptado para el TFG
metrics = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
values = [precision, recall, map50, map50_95]

plt.figure(figsize=(8, 6))
bars = plt.bar(metrics, values, color=['#BF7897', '#78B1BF', '#78BF8B', '#8C78BF'])

# Añadir el valor numérico exacto encima de cada barra
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.01, f'{yval:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Títulos y formato (cambiado a "Test")
plt.title('Testing Metrics for Experiment 3', fontsize=14, pad=15)
plt.ylabel('Value', fontsize=12)
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
best_model = YOLO('runs/detect/train2/weights/best.pt')
test_results = best_model.val(data='data.yaml', split='test')

# Print test metrics
print(f"Test Precision: {test_results.box.mp:.4f}")
print(f"Test Recall: {test_results.box.mr:.4f}")
print(f"Test mAP@50: {test_results.box.map50:.4f}")
print(f"Test mAP@50-95: {test_results.box.map:.4f}")

In [ ]:
# Load the YOLO model
#best_model = YOLO('runs/detect/train3/weights/best.pt')

# Class names 
class_names_english = ['Aluminium foil', 'Bottle cap', 'Bottle', 'Broken glass', 'Can', 'Carton', 
               'Cigarette', 'Cup', 'Lid', 'Other litter', 'Other plastic', 'Paper', 
               'Plastic bag - wrapper', 'Plastic container', 'Pop tab', 'Straw', 
               'Styrofoam piece', 'Unlabeled litter']

class_names = ['Papel de aluminio', 'Tapón de botella', 'Botella', 'Vidrio roto', 'Lata', 'Cartón',
               'Cigarrillo','Vaso', 'Tapa', 'Otros residuos', 'Otros pláticos', 'Papel', 
               'Bolsa de plástico / envoltorio', 'Contenedor de plástico', 'Anilla de lata', 'Pajita', 
               'Corcho blanco', 'Residuo sin clasificar']

# Path to the test images and corresponding labels (annotations)
test_image_dir = 'test/images/'
test_label_dir = 'test/labels/'

# Output directory to save inference results
output_dir = "detection_results/"

testing_detection(best_model,class_names,test_image_dir,test_label_dir,output_dir)

End of analysis

In [ ]:
import os
import pandas as pd

detection_dir = "detection_results3/"

image_rows = []

for file in os.listdir(detection_dir):
    if file.endswith(".txt"):
        path = os.path.join(detection_dir, file)

        with open(path, "r") as f:
            lines = f.readlines()

        pred_areas = []
        pred_classes = []

        mode = None
        width = None
        height = None
        
        for line in lines:
            line = line.strip()

            if "IMAGE SIZE" in line:
                mode = "size"
                continue
                
            # leer las líneas de prediction
            if "PREDICTION" in line:
                mode = "pred"
                continue

            # Saltar la de observation
            if "OBSERVATION" in line:
                mode = None
                continue

            # leer predicciones
            if mode == "size":
                parts = line.split()
                if len(parts) >= 2:
                    try:
                        width = float(parts[0])
                        height = float(parts[1])
                        mode = None  # ya lo hemos leído
                    except:
                        pass
                        
            if mode == "pred":
                parts = line.split()

                if len(parts) == 5:
                    clase, x1, y1, x2, y2 = map(float, parts)

                    area = abs(x2 - x1) * abs(y2 - y1)

                    pred_areas.append(area)
                    pred_classes.append(int(clase))

        # métricas por imagen
        total_area = sum(pred_areas)
        n_residues = len(pred_areas)
        densidad = total_area / (width * height)

        image_rows.append([
            file,
            n_residues,
            pred_areas,
            total_area,
            pred_classes,
            densidad
        ])

# DataFrame final
df_images = pd.DataFrame(image_rows, columns=[
    "Imagen",
    "Número de residuos",
    "Áreas",
    "Área total de residuos",
    "Tipos de residuos",
    "Densidad de residuos (Área de residuos / Área total)"
])

df_images


In [ ]:
from collections import Counter
import pandas as pd

# 1. Diccionario oficial para traducir tus IDs a los nombres reales
class_mapping_ids = {
    0: 'Papel de aluminio', 1: 'Botella', 2: 'Tapón de botella', 
    3: 'Vidrio roto', 4: 'Lata', 5: 'Cartón', 6: 'Cigarrillo',
    7: 'Vaso', 8: 'Tapa', 9: 'Otros residuos', 10: 'Otros plásticos',
    11: 'Papel', 12: 'Bolsa de plástico / envoltorio', 
    13: 'Contenedor de plástico', 14: 'Anilla de lata', 
    15: 'Pajita', 16: 'Corcho blanco', 17: 'Residuo sin clasificar'
}

# 2. Extraer todos los IDs de la columna del DataFrame y aplanarlos en una sola lista
todos_los_ids_predichos = []
for lista_ids in df_images["Tipos de residuos"]:
    todos_los_ids_predichos.extend(lista_ids)

# 3. Contar la frecuencia de cada ID
conteo_ids = Counter(todos_los_ids_predichos)

# 4. Crear un DataFrame limpio con los resultados para verlo ordenado
resumen_detecciones = []
total_detecciones = len(todos_los_ids_predichos)

for clase_id, frecuencia in conteo_ids.items():
    nombre_real = class_mapping_ids.get(clase_id, f"Desconocido ({clase_id})")
    porcentaje = (frecuencia / total_detecciones) * 100
    resumen_detecciones.append([nombre_real, frecuencia, porcentaje])

df_ranking_pred = pd.DataFrame(resumen_detecciones, columns=["Clase de Residuo", "Frecuencia (Detecciones)", "% sobre el Total"])
df_ranking_pred = df_ranking_pred.sort_values(by="Frecuencia (Detecciones)", ascending=False)

print("="*60)
print("   RANKING DE LAS CLASES MÁS DETECTADAS POR EL MODELO")
print("="*6)
print(df_ranking_pred.to_string(index=False, formatters={'% sobre el Total': '{:,.1f}%'.format}))
print("="*60)

In [ ]:
with pd.ExcelWriter("predictions_table.xlsx") as writer:
    df_images.to_excel(writer, sheet_name="predictions", index=False)

In [ ]:
total_pred_residues = df_images["Número de residuos"].sum()
total_pred_area = df_images["Área total de residuos"].sum()

# área total de todas las imágenes
image_area = 416 * 416
total_images = len(df_images)
total_dataset_area = image_area * total_images

global_pred_density = total_pred_area / total_dataset_area

print("----- PREDICTION -----")
print("Total de residuos:", total_pred_residues)
#print("Área total residuos:", total_pred_area)
print("Densidad global:", global_pred_density)

In [ ]:
import os
import pandas as pd

detection_dir = "detection_results3/"

image_rows = []

for file in os.listdir(detection_dir):
    if file.endswith(".txt"):
        path = os.path.join(detection_dir, file)

        with open(path, "r") as f:
            lines = f.readlines()

        obs_areas = []
        obs_classes = []

        mode = None

        for line in lines:
            line = line.strip()

            #  activar OBSERVATION
            if "OBSERVATION" in line:
                mode = "obs"
                continue

            #  leer OBSERVATION
            if mode == "obs":
                parts = line.split()

                if len(parts) == 5:
                    clase, x, y, w, h = map(float, parts)

                    area = w * h  # normalizado (0-1)
                    obs_areas.append(area)
                    obs_classes.append(int(clase))

        #  métricas por imagen
        total_area = sum(obs_areas)
        n_residues = len(obs_areas)

        image_rows.append([
            file,
            n_residues,
            obs_classes,
            total_area
        ])

# 📋 DataFrame final
df_observation = pd.DataFrame(image_rows, columns=[
    "Imagen",
    "Número de residuos",
    "Tipos de residuos",
    "Densidad de residuos"
])

df_observation

In [ ]:
from collections import Counter
import pandas as pd

class_mapping_ids = {
    0: 'Papel de aluminio', 1: 'Botella', 2: 'Tapón de botella', 
    3: 'Vidrio roto', 4: 'Lata', 5: 'Cartón', 6: 'Cigarrillo',
    7: 'Vaso', 8: 'Tapa', 9: 'Otros residuos', 10: 'Otros plásticos',
    11: 'Papel', 12: 'Bolsa de plástico / envoltorio', 
    13: 'Contenedor de plástico', 14: 'Anilla de lata', 
    15: 'Pajita', 16: 'Corcho blanco', 17: 'Residuo sin clasificar'
}

todos_los_ids_reales = []
for lista_ids in df_observation["Tipos de residuos"]:
    todos_los_ids_reales.extend(lista_ids)

conteo_reales = Counter(todos_los_ids_reales)

resumen_reales = []
total_reales = len(todos_los_ids_reales)

for clase_id, frecuencia in conteo_reales.items():
    nombre_real = class_mapping_ids.get(clase_id, f"Desconocido ({clase_id})")
    porcentaje = (frecuencia / total_reales) * 100
    resumen_reales.append([nombre_real, frecuencia, porcentaje])

df_ranking_real = pd.DataFrame(resumen_reales, columns=["Clase de Residuo", "Frecuencia (Real)", "% Real"])

# =================================================================
# Juntamos el ranking Real y el Predicho en una única tabla
# =================================================================
df_comparativo_final = pd.merge(df_ranking_real, df_ranking_pred, on="Clase de Residuo", how="outer").fillna(0)


df_comparativo_final["Frecuencia (Real)"] = df_comparativo_final["Frecuencia (Real)"].astype(int)
df_comparativo_final["Frecuencia (Detecciones)"] = df_comparativo_final["Frecuencia (Detecciones)"].astype(int)

df_comparativo_final = df_comparativo_final.sort_values(by="Frecuencia (Real)", ascending=False)

print("="*85)
print("   TABLA COMPARATIVA FINAL: RECUENTO REAL (GT) VS PREDICCIONES (YOLOv8)")
print("="*85)
print(df_comparativo_final.to_string(index=False, formatters={
    '% Real': '{:,.1f}%'.format,
    '% sobre el Total': '{:,.1f}%'.format
}))
print("="*85)
print(f"Total objetos reales: {total_reales} | Total objetos predichos: {total_detecciones}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

clases = [
    'Bolsa / envoltorio', 'Cigarrillo', 'Vidrio roto', 'Sin clasificar', 
    'Cartón', 'Otros plásticos', 'Botella', 'Vaso', 'Lata', 
    'Tapón', 'Pajita', 'Papel', 'Anilla de lata', 'Otros residuos', 
    'Papel aluminio', 'Corcho blanco', 'Tapa', 'Contenedor plástico'
]

frec_real = [59, 55, 51, 38, 30, 28, 26, 19, 19, 13, 12, 10, 9, 7, 6, 6, 4, 2]
frec_pred = [46, 31, 31, 19, 17, 19, 27, 19, 20, 10, 14, 9, 3, 6, 6, 6, 5, 2]

x = np.arange(len(clases))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))

rects1 = ax.bar(x - width/2, frec_real, width, label='Real', color='#72D69B', alpha=0.9)
rects2 = ax.bar(x + width/2, frec_pred, width, label='Predicho', color='#4682B4', alpha=0.9)

ax.set_ylabel('Frecuencia Absoluta (Unidades)', fontsize=12, fontweight='bold', labelpad=10)
ax.set_title('Comparativo: Distribución de Frecuencias Reales vs. Predicciones', fontsize=14, pad=20, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(clases, rotation=35, ha='right', fontsize=11)

ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.4)

for rect in rects1:
    height = rect.get_height()
    if height > 15:
        ax.annotate(f'{height}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 píxeles de desplazamiento vertical
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, color='black')

for rect in rects2:
    height = rect.get_height()
    if height > 15:
        ax.annotate(f'{height}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, color='black')

fig.tight_layout()
plt.show()

In [ ]:
with pd.ExcelWriter("observations_table.xlsx") as writer:
    df_observation.to_excel(writer, sheet_name="observations", index=False)

In [ ]:
total_obs_residues = df_observation["Número de residuos"].sum()
total_obs_area = df_observation["Densidad de residuos"].sum()

total_images = len(df_observation)

global_obs_density = total_obs_area / total_images

print("----- OBSERVATION -----")
print("Total residuos:", total_obs_residues)
# print("Área total residuos:", total_obs_area)
print("Densidad global:", global_obs_density)

In [ ]:
print("----- COMPARACIÓN -----")
print("Residuos predichos:", total_pred_residues)
print("Residuos reales:", total_obs_residues)

print("Densidad predicha:", global_pred_density)
print("Densidad real:", global_obs_density)

In [ ]:
import pandas as pd

materiales_mapping = {
    'Papel de aluminio': 'Metales/Aluminio',
    'Anilla de lata': 'Metales/Aluminio',
    'Lata': 'Metales/Aluminio',
    
    'Botella': 'Plásticos',
    'Tapón de botella': 'Plásticos',
    'Bolsa de plástico / envoltorio': 'Plásticos',
    'Contenedor de plástico': 'Plásticos',
    'Otros plásticos': 'Plásticos',
    'Pajita': 'Plásticos',
    'Tapa': 'Plásticos',
    'Vaso': 'Plásticos',
    
    'Cartón': 'Papel/Cartón',
    'Papel': 'Papel/Cartón',
    
    'Vidrio roto': 'Vidrio',
    
    'Cigarrillo': 'Colillas/Celulosa',
    
    'Corcho blanco': 'Otros',
    'Otros residuos': 'Otros',
    'Residuo sin clasificar': 'Otros'
}

class_mapping_ids = {
    0: 'Papel de aluminio', 1: 'Botella', 2: 'Tapón de botella', 
    3: 'Vidrio roto', 4: 'Lata', 5: 'Cartón', 6: 'Cigarrillo',
    7: 'Vaso', 8: 'Tapa', 9: 'Otros residuos', 10: 'Otros plásticos',
    11: 'Papel', 12: 'Bolsa de plástico / envoltorio', 
    13: 'Contenedor de plástico', 14: 'Anilla de lata', 
    15: 'Pajita', 16: 'Corcho blanco', 17: 'Residuo sin clasificar'
}

objeto_rows = []

for file in os.listdir(detection_dir):
    if file.endswith(".txt"):
        path = os.path.join(detection_dir, file)
        with open(path, "r") as f:
            lines = f.readlines()
        
        mode = None
        for line in lines:
            line = line.strip()
            if "OBSERVATION" in line:
                mode = "obs"
                continue
            if "IMAGE SIZE" in line or "PREDICTION" in line:
                mode = None
                continue
                
            if mode == "obs":
                parts = line.split()
                if len(parts) == 5:
                    clase_id, x, y, w, h = map(float, parts)
                    nombre_clase = class_mapping_ids[int(clase_id)]
                    macro_material = materiales_mapping[nombre_clase]
                    area_relativa = w * h # Densidad individual del objeto
                    
                    objeto_rows.append([macro_material, area_relativa])

df_objetos = pd.DataFrame(objeto_rows, columns=["Material", "Densidad"])

densidad_total_dataset = df_objetos["Densidad"].sum()

df_analisis_materiales = df_objetos.groupby("Material").agg(
    Cantidad_Unidades=("Densidad", "count"),
    Densidad_Absoluta_Aportada=("Densidad", "sum")
).reset_index()

df_analisis_materiales["% Cantidad (Conteo)"] = (df_analisis_materiales["Cantidad_Unidades"] / len(df_objetos)) * 100
df_analisis_materiales["% Densidad Superficial"] = (df_analisis_materiales["Densidad_Absoluta_Aportada"] / densidad_total_dataset) * 100

print("="*65)
print("   ESTUDIO DE MACRO-CATEGORÍAS DE MATERIALES (PROPUESTA TUTOR)")
print("="*65)
print(df_analisis_materiales.to_string(index=False, formatters={
    'Densidad_Absoluta_Aportada': '{:,.4f}'.format,
    '% Cantidad (Conteo)': '{:,.1f}%'.format,
    '% Densidad Superficial': '{:,.1f}%'.format
}))
print("="*65)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import numpy as np

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 13})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Panel izquierdo: conteo original ---
ax1 = axes[0]
sns.kdeplot(datos_fotos, color="#4A90E2", linewidth=2.5, 
            fill=True, alpha=0.1, bw_adjust=0.7, ax=ax1)
stat1, p1 = stats.shapiro(datos_fotos)
ax1.set_title(f"Conteo de residuos (datos reales)\nShapiro-Wilk: $p < 0.0001$")
ax1.set_xlabel("Número de residuos por imagen")
ax1.set_ylabel("Densidad de probabilidad")
ax1.set_xlim(0, 30)

# --- Panel derecho: densidad superficial original ---
ax2 = axes[1]
sns.kdeplot(datos_densidad * 100, color="#2ECC71", linewidth=2.5,
            fill=True, alpha=0.1, bw_adjust=0.7, ax=ax2)
stat2, p2 = stats.shapiro(datos_densidad)
ax2.set_title(f"Densidad superficial (datos reales)\nShapiro-Wilk: $p < 0.0001$")
ax2.set_xlabel("Porcentaje de suelo ocupado por imagen (%)")
ax2.set_ylabel("Densidad de probabilidad")
ax2.set_xlim(0, 60)
plt.suptitle("Diagnóstico de normalidad sobre los datos originales",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("diagnostico_normalidad.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
ceros = np.sum(datos_densidad == 0)
print(f"Imágenes con densidad cero: {ceros} de {len(datos_densidad)}")
print(f"Mínimo no cero: {datos_densidad[datos_densidad > 0].min():.6f}")

# Probar log puro sin ceros
datos_sin_ceros = datos_densidad[datos_densidad > 0]
log_sin_ceros = np.log(datos_sin_ceros)

stat, p = stats.shapiro(log_sin_ceros)
print(f"\nSin ceros — n = {len(datos_sin_ceros)}")
print(f"Shapiro-Wilk log puro: p = {p:.4f}")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# KDE
import seaborn as sns
sns.kdeplot(log_sin_ceros, color="#2ECC71", linewidth=2.5,
            fill=True, alpha=0.1, bw_adjust=0.7, ax=axes[0])
axes[0].set_title(f"log(densidad) sin ceros\nShapiro-Wilk p = {p:.4f}")
axes[0].set_xlabel("log(Densidad superficial)")

# QQ-plot
stats.probplot(log_sin_ceros, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot: log(densidad) sin ceros")

plt.tight_layout()
plt.show()

In [ ]:
log_densidad = np.log(datos_densidad)

n = len(log_densidad)
media_log = np.mean(log_densidad)
s_log = np.std(log_densidad, ddof=1)
t_crit = stats.t.ppf(0.975, df=n-1)
margen = t_crit * (s_log / np.sqrt(n))

li_log = media_log - margen
ls_log = media_log + margen

# Transformación inversa
media_original = np.exp(media_log)
li_original = np.exp(li_log)
ls_original = np.exp(ls_log)

print(f"n:                    {n}")
print(f"Media log:            {media_log:.4f}")
print(f"Desv. típica log:     {s_log:.4f}")
print(f"t crítico:            {t_crit:.4f}")
print(f"IC log:               [{li_log:.4f}, {ls_log:.4f}]")
print(f"Media geométrica:     {media_original*100:.2f}%")
print(f"IC original:          [{li_original*100:.2f}%, {ls_original*100:.2f}%]")

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 13})

log_densidad = np.log(datos_densidad)
media_log_def = -3.2979
li_log_def = -3.6239
ls_log_def = -2.9718

plt.figure(figsize=(8, 4.5))
sns.kdeplot(log_densidad, color="#2ECC71", linewidth=2.5,
            fill=True, alpha=0.1, bw_adjust=0.7)
plt.axvline(media_log_def, color="#D0021B", linestyle="--", linewidth=2,
            label=f"Media geométrica (3.70%)")
plt.axvspan(li_log_def, ls_log_def, color="#D0021B", alpha=0.18,
            label=r"IC 95% para $\mu_{den}$ ([2.67%, 5.12%])")
plt.title("Estimación de la Media de la Densidad Superficial por Imagen\n"
          r"(escala $\log(D_s)$)")
plt.xlabel(r"$\log$(Densidad superficial)")
plt.ylabel("Densidad de probabilidad")
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig("inferencia_densidad_log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import scipy.stats as stats

datos_densidad = df_observation["Densidad de residuos"].to_numpy()
n_imagenes = len(datos_densidad)

# --- Transformación logarítmica ---
# Añadimos un epsilon pequeño para evitar log(0) si alguna imagen tiene densidad 0
epsilon = 1e-6
datos_den_log = np.log(datos_densidad + epsilon)

media_den_log    = np.mean(datos_den_log)
varianza_den_log = np.var(datos_den_log, ddof=1)
desviacion_den_log = np.sqrt(varianza_den_log)

confianza = 0.95
z_critico = stats.norm.ppf((1 + confianza) / 2)
margen_error_den_log = z_critico * (desviacion_den_log / np.sqrt(n_imagenes))

# IC en escala log
li_den_log = media_den_log - margen_error_den_log
ls_den_log = media_den_log + margen_error_den_log

# Volver a escala original
media_den_original = np.exp(media_den_log)
li_den_original    = np.exp(li_den_log)
ls_den_original    = np.exp(ls_den_log)

# Media aritmética real (para referencia)
media_aritmetica_den = np.mean(datos_densidad)

print("="*60)
print("   INFERENCIA DE DENSIDAD SUPERFICIAL (ESCALA LOG)")
print("="*60)
print(f"Número de imágenes analizadas (n):              {n_imagenes}")
print(f"Media aritmética muestral (referencia):         {media_aritmetica_den:.6f} ({media_aritmetica_den*100:.2f}%)")
print(f"Media geométrica (exp(media_log)):              {media_den_original:.6f} ({media_den_original*100:.2f}%)")
print(f"Media en escala log:                            {media_den_log:.4f}")
print(f"Desviación típica en escala log:                {desviacion_den_log:.4f}")
print("-"*60)
print(f"IC al 95% en escala log:  [{li_den_log:.4f} , {ls_den_log:.4f}]")
print(f"IC al 95% en escala original (proporción):")
print(f"  [{li_den_original:.6f} , {ls_den_original:.6f}]")
print(f"IC al 95% en porcentaje de suelo ocupado:")
print(f"  [{li_den_original*100:.2f}% , {ls_den_original*100:.2f}%]")
print("="*60)

In [ ]:
import scipy.stats as stats
import matplotlib.pyplot as plt

# Test de normalidad sobre la densidad original
stat, p = stats.shapiro(datos_densidad)
print(f"Test de Shapiro-Wilk sobre densidad original: p = {p:.4f}")

stat_log, p_log = stats.shapiro(np.log(datos_densidad + 1e-6))
print(f"Test de Shapiro-Wilk sobre log(densidad):     p = {p_log:.4f}")

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

transformaciones = {
    "Original": datos_densidad,
    "log(x + ε)": np.log(datos_densidad + 1e-6),
    "sqrt(x)": np.sqrt(datos_densidad),
    "x^(1/3)": np.cbrt(datos_densidad)
}

for ax, (nombre, datos) in zip(axes.flatten(), transformaciones.items()):
    stat, p = stats.shapiro(datos)
    ax.hist(datos, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f"{nombre}  |  Shapiro p = {p:.4f}")
    ax.set_xlabel("Valor transformado")
    ax.set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

# Imprimir p-valores
print("\nResumen de p-valores (Shapiro-Wilk):")
for nombre, datos in transformaciones.items():
    stat, p = stats.shapiro(datos)
    print(f"  {nombre:15s}: p = {p:.4f}  {'✓ más normal' if p > 0.05 else '✗'}")